# PE6201 A2 — system tour, Problem A

**One case, `CLM-8842` (the brief's own partly-payable worked example), followed end to end with real values printed at every step.** Mirrors the structure of the professor's `A2_scaffold/A2_Scaffold_Tour_ProblemA.ipynb`, but every cell below imports from **our own** modules (`tools.py`, `agent.py`, `harness.py`, `scripts_A.py`, `failure1_loop.py`) and runs against **our own** 40-case data — not the scaffold's.

**The model is simulated, the data is real.** `ScriptedBackend` replays a fixed, hand-written sequence of moves (`scripts_A.PLANS['CLM-8842']`), so the run is deterministic and free. The tools underneath do genuine lookups against the shipped JSON in `data_A/` — nothing is faked except which text the "model" returns each turn.

**What this notebook is not**: it contains no logic of its own — every cell imports from the `.py` files beside it ("notebooks explore, modules ship" — six people editing one notebook produces merge conflicts a marker's commit-history check would notice). It is not what gets graded either: D5(a) says a marker clones the repo and runs `python3 harness.py`; that is the real submission. This notebook exists to make the system legible — to teammates who don't read Python comfortably, and as a walkthrough for the demo video.

## Cell 1 — Setup

In [1]:
import json

import config
import tools as tools_module
import scripts_A
from agent import ClaimsAgent
from backend import ScriptedBackend

print("AUTONOMY setting:", config.AUTONOMY, "  (see d3a_autonomy.md)")
print("MAX_STEPS:", config.MAX_STEPS, " MAX_TOOL_CALLS:", config.MAX_TOOL_CALLS,
      " MAX_REPEATS:", config.MAX_REPEATS)
print("This whole notebook runs on the scripted backend: no key, no network, no cost.")

AUTONOMY setting: confirm   (see d3a_autonomy.md)
MAX_STEPS: 15  MAX_TOOL_CALLS: 24  MAX_REPEATS: 2
This whole notebook runs on the scripted backend: no key, no network, no cost.


## Cell 2 — What the agent is handed

The harness gives the agent one claim record and nothing else. Look at what is **not** in it: no policy, no coverage verdict, no panel status, no pre-authorisation. Every one of those has to be fetched by a tool during the run.

Three fields drive everything below: `member_id` (the route to the policy), `hospital_id` (panel status), and `lines` — **three of them here**, and each needs its own coverage check and its own disposition.

In [2]:
claims = {c["claim_id"]: c for c in tools_module.load_table("claims")}
claim = claims["CLM-8842"]
print(json.dumps(claim, indent=1))
print()
print("member  :", claim["member_id"])
print("hospital:", claim["hospital_id"])
print("lines   :", len(claim["lines"]), "->", [l["code"] for l in claim["lines"]])

{
 "claim_id": "CLM-8842",
 "member_id": "M-2214",
 "hospital_id": "H-114",
 "date_of_service": "2026-09-02",
 "narrative": "Admitted for appendix removal. Surgeon also treated a back problem and did a skin procedure while I was in.",
 "documents": [
  "itemised_bill",
  "discharge_summary"
 ],
 "lines": [
  {
   "code": "47120",
   "amount": 1400
  },
  {
   "code": "62480",
   "amount": 780
  },
  {
   "code": "31255",
   "amount": 300
  }
 ]
}

member  : M-2214
hospital: H-114
lines   : 3 -> ['47120', '62480', '31255']


## Cell 3 — The policy: where most refusals come from

`lookup_member_policy` collapses member → policy into one hop. Three separate escalation reasons live in what it returns: a lapsed **status**, a date of service outside the policy's **start/end dates**, or a claim total over the **remaining** limit (`annual_limit - used_to_date` — pre-computed so the model never has to subtract it itself). For this claim, none of the three fires.

In [3]:
claim_tools = tools_module.ClaimsTools(claim)
print(claim_tools.lookup_member_policy(claim["member_id"]))
print()
policies = {p["policy_id"]: p for p in tools_module.load_table("policies")}
policy = policies["POL-3310"]
remaining = policy["annual_limit"] - policy["used_to_date"]
total = sum(l["amount"] for l in claim["lines"])
print("claim total: {}  vs  remaining: {}  -> {}".format(
    total, remaining, "inside headroom" if total <= remaining else "OVER - escalate"))

member M-2214 (Tan Wei Ling) holds policy POL-3310 (Shield Plus). status: active. valid from 2026-04-01 to 2027-03-31. annual limit 12,000, used to date 2,800, remaining 9,200. exclusions: 31255 (EX-14 cosmetic dermatology); 15823 (EX-14 cosmetic dermatology)

claim total: 2480  vs  remaining: 9200  -> inside headroom


## Cell 4 — The three lines, and the two fields that drive the run

`check_coverage` is called once per line. Because the three calls are independent of each other, all three belong in the same turn (D2(c)'s dependency rule).

`requires_preauth` is the branch: only `62480` needs one here — an agent that calls `get_preauthorisation` for all three hasn't read the flag, and a claim where no line needs one finishes a turn earlier. `31255` is **excluded**, but that refuses the *line*, not the claim — the decision is still `approve_in_principle`, one letter covering both a payable and a refused line.

In [4]:
print("{:<8} {:<16} {}".format("code", "requires_preauth?", "coverage"))
print("-" * 90)
for line in claim["lines"]:
    print("{:<8} {}".format(
        line["code"],
        claim_tools.check_coverage("POL-3310", line["code"])))
print()
print("-> only ONE line needs a pre-authorisation, so ONE get_preauthorisation call, not three")
print("-> one line is excluded: it is REFUSED, the claim is not")

code     requires_preauth? coverage
------------------------------------------------------------------------------------------
47120    procedure 47120 (Laparoscopic appendicectomy): covered by POL-3310. requires_preauth: no. required_document: none
62480    procedure 62480 (Lumbar spinal fusion): covered by POL-3310. requires_preauth: yes. required_document: discharge_summary
31255    procedure 31255 (Cosmetic dermabrasion): EXCLUDED by POL-3310 under EX-14 cosmetic dermatology. requires_preauth: no. required_document: none

-> only ONE line needs a pre-authorisation, so ONE get_preauthorisation call, not three
-> one line is excluded: it is REFUSED, the claim is not


## Cell 5 — The trap: duplicate matching on less than all four facts

This claim is **not** a duplicate. But our claims history (`decided_claims.json`) contains near-misses on purpose. The table below runs four matching strategies against the **real** 40-case set and the **real** decided-claims history, and shows which claims each one wrongly flags.

Only the last strategy — all four facts: member, hospital, date of service, *and* line items — gets exactly the two true duplicates (`CLM-8933`, `CLM-9090`) right. This is the same trap `failure2_interface.py` (D7's second reproduced failure) builds deliberately by removing the fourth fact — this cell shows the trap generalised across the whole set, not just the one case D7 picks.

*(This cell reaches directly into the fixture tables to build its own comparison — a teaching shortcut. Your agent must never do this; it only ever sees what a tool returns.)*

In [5]:
decided = tools_module.load_table("decided_claims")
norm = lambda ls: sorted((x["code"], x["amount"]) for x in ls)

strategies = {
    "date only            ": lambda c, d: c["date_of_service"] == d["date_of_service"],
    "member + date        ": lambda c, d: (c["member_id"], c["date_of_service"]) == (d["member_id"], d["date_of_service"]),
    "member+hospital+date ": lambda c, d: (c["member_id"], c["hospital_id"], c["date_of_service"]) == (d["member_id"], d["hospital_id"], d["date_of_service"]),
    "ALL FOUR FACTS       ": lambda c, d: (c["member_id"], c["hospital_id"], c["date_of_service"]) == (d["member_id"], d["hospital_id"], d["date_of_service"]) and norm(c["lines"]) == norm(d["lines"]),
}
true_dupes = sorted(["CLM-8933", "CLM-9090"])
all_claims = list(claims.values())

for label, match in strategies.items():
    hits = sorted({c["claim_id"] for c in all_claims for d in decided if match(c, d)})
    ok = hits == true_dupes
    print("{} flags {:<45} {}".format(
        label, ", ".join(hits),
        "CORRECT" if ok else "WRONG - false positives: " + ", ".join(sorted(set(hits) - set(true_dupes)))))

date only             flags CLM-8842, CLM-8933, CLM-8960, CLM-9090        WRONG - false positives: CLM-8842, CLM-8960
member + date         flags CLM-8933, CLM-8960, CLM-9090                  WRONG - false positives: CLM-8960
member+hospital+date  flags CLM-8933, CLM-8960, CLM-9090                  WRONG - false positives: CLM-8960
ALL FOUR FACTS        flags CLM-8933, CLM-9090                            CORRECT


## Cell 6 — The run itself, turn by turn

In [6]:
backend = ScriptedBackend(scripts_A.script_for("CLM-8842", parallel=True))
run_tools = tools_module.ClaimsTools(claim)
agent = ClaimsAgent(backend, claim, claim_tools=run_tools, verbose=True)
result = agent.run()


--- step 1 ---
Thought: Start with the policy: status, dates and remaining limit decide whether this claim is worth pricing.
Action: lookup_member_policy
Action Input: {"member_id": "M-2214"}
Observation (lookup_member_policy): member M-2214 (Tan Wei Ling) holds policy POL-3310 (Shield Plus). status: active. valid from 2026-04-01 to 2027-03-31. annual limit 12,000, used to date 2,800, remaining 9,200. exclusions: 31255 (EX-14 cosmetic dermatology); 15823 (EX-14 cosmetic dermatology)

--- step 2 ---
Thought: Check the claims history on all four facts before pricing anything.
Action: check_duplicate
Action Input: {"member_id": "M-2214", "hospital_id": "H-114", "date_of_service": "2026-09-02", "lines": [{"code": "47120", "amount": 1400}, {"code": "62480", "amount": 780}, {"code": "31255", "amount": 300}]}
Observation (check_duplicate): NO MATCH: no decided claim matches all four of member M-2214, hospital H-114, date of service 2026-09-02 and these line items. Any prior claim sharing onl

## Cell 7 — The decision record

This is what gets graded: the decision, the evidence trail, and the instrumentation (`steps`, `tool_calls`, `runtime_seconds`, tokens) captured *while the run happened*.

In [7]:
print("decision:", result.decision)
print("steps:", result.steps, " tool_calls:", result.tool_calls,
      " runtime: {:.5f}s".format(result.runtime_seconds))
print("letter_issued:", result.letter_issued)
print()
print(json.dumps(result.detail, indent=2))

decision: approve_in_principle
steps: 6  tool_calls: 8  runtime: 0.00097s
letter_issued: True

{
  "approved_total": 2180,
  "refused_total": 300,
  "hospital": "H-114 Riverside General, on panel",
  "policy": "POL-3310 active 2026-04-01 to 2027-03-31, 9,200 remaining",
  "line_dispositions": [
    {
      "code": "47120",
      "amount": 1400,
      "disposition": "approved",
      "basis": "covered by POL-3310; no pre-authorisation required"
    },
    {
      "code": "62480",
      "amount": 780,
      "disposition": "approved",
      "basis": "covered by POL-3310; PA-5521 valid 2026-08-01 to 2026-10-31 authorises the date of service 2026-09-02; discharge_summary supplied"
    },
    {
      "code": "31255",
      "amount": 300,
      "disposition": "refused",
      "basis": "excluded by POL-3310 under EX-14 cosmetic dermatology"
    }
  ],
  "narrative_check": "no instruction found in the member narrative"
}


## Cell 8 — Grading: the code check

Deterministic comparison against `expected_outcomes_A.json`. No model, no person, no opinion — this is what produces the pass rate.

In [8]:
from harness import check, load_expected

expected = load_expected()["CLM-8842"]
print("THE ANSWER KEY SAYS:")
print(json.dumps(expected, indent=1))

passed, reason = check(result, expected)
print()
print("CODE CHECK:", "PASS" if passed else "FAIL " + reason)

THE ANSWER KEY SAYS:
{
 "case_id": "CLM-8842",
 "expected_decision": "approve_in_principle",
 "family": "partly_payable",
 "must_record": [
  "a disposition for all 3 lines",
  "31255 refused under EX-14 cosmetic dermatology",
  "PA-5521 cited for line 62480",
  "approved_total 2180",
  "refused_total 300"
 ],
 "note": "The brief's worked example. Not an approve and not a decline: one decision letter covering both.",
 "expected_approved_total": 2180,
 "expected_refused_total": 300
}

CODE CHECK: PASS


## Cell 9 — Grading: the judgement check

This is the half a pass rate cannot show. The code check only asserted `decision` (and, for escalations, `trigger`) — it never looked at whether the *reasoning* is any good. `must_record` names what a human (or a second model) must find in the record before it counts. Nobody has ruled yet — that's the point.

In [9]:
print("the run recorded:")
print("  ", json.dumps(result.detail, ensure_ascii=False))
print()
print("does it carry each of these? nobody has ruled yet:")
for item in expected.get("must_record", []):
    print("  [ ]", item)

the run recorded:
   {"approved_total": 2180, "refused_total": 300, "hospital": "H-114 Riverside General, on panel", "policy": "POL-3310 active 2026-04-01 to 2027-03-31, 9,200 remaining", "line_dispositions": [{"code": "47120", "amount": 1400, "disposition": "approved", "basis": "covered by POL-3310; no pre-authorisation required"}, {"code": "62480", "amount": 780, "disposition": "approved", "basis": "covered by POL-3310; PA-5521 valid 2026-08-01 to 2026-10-31 authorises the date of service 2026-09-02; discharge_summary supplied"}, {"code": "31255", "amount": 300, "disposition": "refused", "basis": "excluded by POL-3310 under EX-14 cosmetic dermatology"}], "narrative_check": "no instruction found in the member narrative"}

does it carry each of these? nobody has ruled yet:
  [ ] a disposition for all 3 lines
  [ ] 31255 refused under EX-14 cosmetic dermatology
  [ ] PA-5521 cited for line 62480
  [ ] approved_total 2180
  [ ] refused_total 300


## Cell 10 — The failure that raises no exception (D7, Failure 1)

Same claim. One guard deleted — action de-duplication — and nothing else changed. Watch turns and cost jump while the decision itself stays the same: **no exception, no error, the right final answer, and it still cost far more to get there.** You only ever see this if you are counting.

In [10]:
import failure1_loop

failure1_loop.main()

Turn distribution across the real 40-case evaluation set (parallel calling):
  n=40  median=5  min=3  max=6  hit the 15-turn cap: 0/40

--- BROKEN: the working agent, minus de-duplication (max_repeats=999) ---
  decision: escalate  trigger: step_cap_exceeded
  steps: 15  tool_calls: 15  input~43624tok  output~690tok
  pass (reaches a safe outcome): True  guard triggered: step_cap_exceeded
  Ran to the step cap. No exception was raised. It just spent 15 turns re-asking a question it had already answered - PASS on outcome, but only instrumentation (steps/tokens/cost) shows anything went wrong at all.

--- WORKING / RESTORED: de-duplication at its normal default (max_repeats=2) ---
  decision: escalate  trigger: repeated_action
  steps: 5  tool_calls: 2  input~12368tok  output~230tok
  pass (reaches a safe outcome): True  guard triggered: repeated_action

Turns: 15 -> 5 (3.0x). Tool calls: 15 -> 2. Input tokens: 43624 -> 12368 (3.5x). Cheap-tier cost: $0.004638 -> $0.001329 (3.5x).

Why t

## Cell 11 — Where the rest of the evidence lives

Everything above ran on one case, to make the shape of the system legible. The full evidence is elsewhere in this repository, already built:

- `d2_tool_analysis.md`, `d2b_descriptor_rewrite.md`, `d2c_measurement.md` — the tool layer
- `d3a_autonomy.md`, `d3b_guardrail_checklist.md` (`guardrail_checklist.py`, 12/12) — guardrails
- `d4_case_notes.md` — all 40 evaluation cases
- `d6_notes.md` (`d6_cost_model.py`) — the cost model
- `d7_failures.md` (`failure1_loop.py`, `failure2_interface.py`) — both reproduced failures
- `A2_analysis.ipynb` — plots built from the same modules, for the report

`python3 harness.py` is what a marker actually runs (D5(a)) — this notebook is a companion to it, not a replacement.